### Test Flush

In [1]:
!pip install kafka-python

In [2]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

topic = "air_quality_stream"

print("Producer connected")

Producer connected


In [3]:
message = {
    "location_id": 31,
    "parameter": "pm25",
    "value": 12.4
}

producer.send(topic, message)
producer.flush()

print("Message sent")

Message sent


In [4]:
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

data = {
    "location_id": 31,
    "pm25": 18.5
}

producer.send("air_quality_stream", data)
producer.flush()

print("Message sent")

Message sent


### Code

In [5]:
!pip install requests

In [22]:
import boto3
import json
import requests
from datetime import datetime

In [7]:
bucket_name = "openaq-project-dm"
raw_prefix = "raw/locations/"
latest_prefix = "latest/"

In [8]:
s3 = boto3.client("s3")
print("Connected to S3")

Connected to S3


In [9]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=raw_prefix
)

In [10]:
files = []

In [11]:
for obj in response.get("Contents", []):

    # Extract the file key (full path in S3)
    key = obj["Key"]

    # Only process JSON files
    if key.endswith(".json"):
        files.append(key)

print(f"Found {len(files)} JSON files in raw/locations/")

Found 16 JSON files in raw/locations/


In [12]:
location_ids = set()

In [13]:
for file_key in files:

    print(f"Reading file: {file_key}")

    # Download the file from S3
    obj = s3.get_object(
        Bucket=bucket_name,
        Key=file_key
    )

Reading file: raw/locations/page_1.json
Reading file: raw/locations/page_10.json
Reading file: raw/locations/page_11.json
Reading file: raw/locations/page_12.json
Reading file: raw/locations/page_13.json
Reading file: raw/locations/page_14.json
Reading file: raw/locations/page_15.json
Reading file: raw/locations/page_16.json
Reading file: raw/locations/page_2.json
Reading file: raw/locations/page_3.json
Reading file: raw/locations/page_4.json
Reading file: raw/locations/page_5.json
Reading file: raw/locations/page_6.json
Reading file: raw/locations/page_7.json
Reading file: raw/locations/page_8.json
Reading file: raw/locations/page_9.json


In [14]:
data = json.loads(obj["Body"].read())

In [15]:
results = data.get("results", [])

In [17]:
    for item in results:

        # Extract the location id
        if isinstance(item, dict):

            location_id = item.get("id")

            # Only add numeric IDs
            if isinstance(location_id, int):
                location_ids.add(location_id)

In [18]:
print(f"Unique location IDs found: {len(location_ids)}")

Unique location IDs found: 100


In [19]:
location_ids = list(location_ids)

In [20]:
all_api_results = []

In [ ]:
for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    print(f"Fetching data for location {loc_id}")

    response = requests.get(url)

    api_data = response.json()

    results = api_data.get("results", [])

    all_api_results.extend(results)


print(f"Total records collected: {len(all_api_results)}")

Fetching data for location 913
Fetching data for location 914
Fetching data for location 915
Fetching data for location 916
Fetching data for location 917
Fetching data for location 918
Fetching data for location 919
Fetching data for location 920
Fetching data for location 921
Fetching data for location 922
Fetching data for location 923
Fetching data for location 924
Fetching data for location 925
Fetching data for location 926
Fetching data for location 927
Fetching data for location 928
Fetching data for location 929
Fetching data for location 930
Fetching data for location 931
Fetching data for location 932
Fetching data for location 933
Fetching data for location 935
Fetching data for location 936
Fetching data for location 937
Fetching data for location 938
Fetching data for location 939
Fetching data for location 940
Fetching data for location 941
Fetching data for location 942
Fetching data for location 944
Fetching data for location 945
Fetching data for location 946
Fetching

In [ ]:
output_data = {

    "generated_at": datetime.utcnow().isoformat(),

    "location_ids": location_ids
}

C:\Users\admin\AppData\Local\Temp\ipykernel_17792\1054324449.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


In [25]:
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

# Create S3 path for output file
s3_key = f"{latest_prefix}unique_location_ids_{timestamp}.json"

C:\Users\admin\AppData\Local\Temp\ipykernel_17792\2016513290.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


In [26]:
s3.put_object(
    Bucket=bucket_name,
    Key=s3_key,
    Body=json.dumps(output_data)
)

print("Upload completed")

print(f"S3 location: s3://{bucket_name}/{s3_key}")

Upload completed
S3 location: s3://openaq-project-dm/latest/unique_location_ids_20260307_150217.json


In [ ]:
import boto3
import json
import requests
from datetime import datetime

bucket_name = "openaq-project-dm"
raw_prefix = "raw/locations/"
latest_prefix = "latest/"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=raw_prefix
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

print("Files found:", len(files))

location_ids = set()

for file_key in files:

    obj = s3.get_object(
        Bucket=bucket_name,
        Key=file_key
    )

    data = json.loads(obj["Body"].read())

    for item in data.get("results", []):
        loc_id = item.get("id")

        if isinstance(loc_id, int):
            location_ids.add(loc_id)

print("Unique IDs:", len(location_ids))

all_results = []

for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    print("Calling API:", url)

    r = requests.get(url)

    api_data = r.json()

    all_results.extend(api_data.get("results", []))


print("Total API records:", len(all_results))

output = {
    "generated_at": datetime.utcnow().isoformat(),
    "records": all_results
}

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

key = f"{latest_prefix}openaq_latest_{timestamp}.json"

s3.put_object(
    Bucket=bucket_name,
    Key=key,
    Body=json.dumps(output)
)

print("Uploaded:", key)

Files found: 16
Unique IDs: 1600
Calling API: https://api.openaq.org/v3/locations/3/latest
Calling API: https://api.openaq.org/v3/locations/4/latest
Calling API: https://api.openaq.org/v3/locations/5/latest
Calling API: https://api.openaq.org/v3/locations/6/latest
Calling API: https://api.openaq.org/v3/locations/7/latest
Calling API: https://api.openaq.org/v3/locations/8/latest
Calling API: https://api.openaq.org/v3/locations/9/latest
Calling API: https://api.openaq.org/v3/locations/10/latest
Calling API: https://api.openaq.org/v3/locations/11/latest
Calling API: https://api.openaq.org/v3/locations/12/latest
Calling API: https://api.openaq.org/v3/locations/13/latest
Calling API: https://api.openaq.org/v3/locations/14/latest
Calling API: https://api.openaq.org/v3/locations/15/latest
Calling API: https://api.openaq.org/v3/locations/16/latest
Calling API: https://api.openaq.org/v3/locations/17/latest
Calling API: https://api.openaq.org/v3/locations/18/latest
Calling API: https://api.opena

C:\Users\admin\AppData\Local\Temp\ipykernel_17792\3298143788.py:82: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),
C:\Users\admin\AppData\Local\Temp\ipykernel_17792\3298143788.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Uploaded: latest/openaq_latest_20260307_151456.json


In [ ]:
import boto3
import json
import requests
from datetime import datetime

bucket_name = "openaq-project-dm"
raw_prefix = "raw/locations/"
latest_prefix = "raw/latest/"

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix=raw_prefix
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

print("Files found:", len(files))

location_ids = set()

for file_key in files:

    obj = s3.get_object(
        Bucket=bucket_name,
        Key=file_key
    )

    data = json.loads(obj["Body"].read())

    results = data.get("results", [])

    for item in results:

        loc_id = item.get("id")

        if isinstance(loc_id, int):
            location_ids.add(loc_id)


print("Unique location IDs:", len(location_ids))

all_results = []

for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    print("Calling:", url)

    r = requests.get(url)

    api_data = r.json()

    api_results = api_data.get("results", [])

    if api_results:
        all_results.extend(api_results)


print("Total API records collected:", len(all_results))

output = {
    "generated_at": datetime.utcnow().isoformat(),
    "records": all_results
}

timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

key = f"{latest_prefix}openaq_latest_{timestamp}.json"

s3.put_object(
    Bucket=bucket_name,
    Key=key,
    Body=json.dumps(output)
)

print("Uploaded file:", key)

Files found: 16
Unique location IDs: 1600
Calling: https://api.openaq.org/v3/locations/3/latest
Calling: https://api.openaq.org/v3/locations/4/latest
Calling: https://api.openaq.org/v3/locations/5/latest
Calling: https://api.openaq.org/v3/locations/6/latest
Calling: https://api.openaq.org/v3/locations/7/latest
Calling: https://api.openaq.org/v3/locations/8/latest
Calling: https://api.openaq.org/v3/locations/9/latest
Calling: https://api.openaq.org/v3/locations/10/latest
Calling: https://api.openaq.org/v3/locations/11/latest
Calling: https://api.openaq.org/v3/locations/12/latest
Calling: https://api.openaq.org/v3/locations/13/latest
Calling: https://api.openaq.org/v3/locations/14/latest
Calling: https://api.openaq.org/v3/locations/15/latest
Calling: https://api.openaq.org/v3/locations/16/latest
Calling: https://api.openaq.org/v3/locations/17/latest
Calling: https://api.openaq.org/v3/locations/18/latest
Calling: https://api.openaq.org/v3/locations/19/latest
Calling: https://api.openaq.or

C:\Users\admin\AppData\Local\Temp\ipykernel_17792\1040680683.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),
C:\Users\admin\AppData\Local\Temp\ipykernel_17792\1040680683.py:97: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Uploaded file: raw/latest/openaq_latest_20260307_152703.json


In [ ]:
import boto3
import json
import requests

bucket = "openaq-project-dm"
raw_prefix = "raw/locations/"
latest_prefix = "raw/latest/"

headers = {
    "X-API-Key": "edec33bcd3cc16aa693bb1acd3bf40be5a462841c57049dbdca3fab2ac02a928"
}

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=raw_prefix
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

print("Files found:", len(files))


location_ids = set()

for file in files:

    obj = s3.get_object(Bucket=bucket, Key=file)

    data = json.loads(obj["Body"].read())

    for r in data.get("results", []):

        if "id" in r:
            location_ids.add(r["id"])

print("Unique location IDs:", len(location_ids))

all_results = []

for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    print("Calling:", url)

    try:

        r = requests.get(url, headers=headers)

        # print HTTP status
        print("Status:", r.status_code)

        if r.status_code != 200:
            print("API ERROR:", r.text)
            continue

        api_json = r.json()

        results = api_json.get("results", [])

        all_results.extend(results)

    except Exception as e:
        print("REQUEST FAILED:", e)


print("Total records collected:", len(all_results))

output = {
    "results": all_results
}

s3.put_object(
    Bucket=bucket,
    Key=f"{latest_prefix}openaq_latest.json",
    Body=json.dumps(output)
)

print("Upload complete")

Files found: 16
Unique location IDs: 1600
Calling: https://api.openaq.org/v3/locations/3/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/4/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/5/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/6/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/7/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/8/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/9/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/10/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/11/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/12/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/13/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/14/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/15/latest
Status: 200
Calling: https://api.openaq.org/v3/locations/16/latest
Status: 200
Calling: https://api.openaq

In [1]:
!pip install kafka-python boto3 requests

In [ ]:
from kafka import KafkaProducer
import boto3
import json
import requests

API_KEY = "MY_API_KEY"

headers = {"X-API-Key": API_KEY}

bucket = "openaq-project-dm"
raw_prefix = "raw/locations/"

topic = "air_quality_stream"

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

s3 = boto3.client("s3")


response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=raw_prefix
)

files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".json")]

location_ids = set()

for file in files:

    obj = s3.get_object(Bucket=bucket, Key=file)

    data = json.loads(obj["Body"].read())

    for r in data.get("results", []):
        if "id" in r:
            location_ids.add(r["id"])

print("Locations:", len(location_ids))


for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    try:

        r = requests.get(url, headers=headers)

        if r.status_code != 200:
            print("API error:", r.text)
            continue

        api_data = r.json()

        for record in api_data.get("results", []):

            producer.send(topic, record)

    except Exception as e:
        print("Producer error:", e)

producer.flush()

print("Producer finished")

Locations: 1600
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detail":"Too many requests"}
API error: {"detai

KeyboardInterrupt: 

In [ ]:
from kafka import KafkaProducer
import boto3
import json
import requests
import time

API_KEY = "MY_API_KEY"

headers = {
    "X-API-Key": API_KEY
}

bucket = "openaq-project-dm"
raw_prefix = "raw/locations/"
topic = "air_quality_stream"

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

s3 = boto3.client("s3")


response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=raw_prefix
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

print("Raw files found:",      len(files))

location_ids = set()

for file in files:

    obj = s3.get_object(Bucket=bucket, Key=file)

    data = json.loads(obj["Body"].read())

    for r in data.get("results", []):
        if "id" in r:
            location_ids.add(r["id"])

print("Unique location IDs:", len(location_ids))

for loc_id in location_ids:

    url = f"https://api.openaq.org/v3/locations/{loc_id}/latest"

    print("Calling API:", url)

    retries = 3

    while retries > 0:

        try:

            r = requests.get(url, headers=headers)

            if r.status_code == 429:
                print("Rate limit hit. Sleeping 10 seconds...")
                time.sleep(10)
                retries -= 1
                continue

            if r.status_code != 200:
                print("API error:", r.text)
                break

            api_data = r.json()

            results = api_data.get("results", [])

            for record in results:
                producer.send(topic, record)

            print("Records sent:", len(results))

            break

        except Exception as e:
            print("Request failed:", e)
            retries -= 1
            time.sleep(5)

    time.sleep(1)

producer.flush()

print("Producer finished streaming data to Kafka")

Raw files found: 16
Unique location IDs: 1600
Calling API: https://api.openaq.org/v3/locations/3/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/4/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/5/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/6/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/7/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/8/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/9/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/10/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/11/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/12/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/13/latest
Records sent: 2
Calling API: https://api.openaq.org/v3/locations/14/latest
Records sent: 0
Calling API: https://api.openaq.org/v3/locations/15/latest
Re

In [ ]:
from kafka import KafkaConsumer
import boto3
import json
import datetime

topic = "air_quality_stream"

bucket = "openaq-project-dm"
prefix = "raw/latest/"

batch_size = 500

consumer = KafkaConsumer(
    topic,
    bootstrap_servers="localhost:9092",
    value_deserializer=lambda x: json.loads(x.decode("utf-8")),
    auto_offset_reset="earliest",
    enable_auto_commit=True
)

s3 = boto3.client("s3")

print("Consumer started")

buffer = []
seen = set()

for message in consumer:

    record = message.value

    try:

        unique_key = f"{record['locationsId']}_{record['sensorsId']}_{record['datetime']['utc']}"

        if unique_key in seen:
            continue

        seen.add(unique_key)

        buffer.append(record)

        print("Buffered record:", unique_key)

    except Exception as e:
        print("Record parse error:", e)
        continue

    if len(buffer) >= batch_size:

        timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")

        key = f"{prefix}openaq_{timestamp}.json"

        try:

            s3.put_object(
                Bucket=bucket,
                Key=key,
                Body=json.dumps({"results": buffer})
            )

            print("Uploaded batch:", key)

            buffer = []

        except Exception as e:

            print("S3 upload error:", e)

Consumer started
Record parse error: 'locationsId'
Record parse error: 'locationsId'
Record parse error: 'locationsId'
Record parse error: 'locationsId'
Record parse error: 'locationsId'
Buffered record: 13_13866_2018-02-22T04:00:00Z
Buffered record: 13_13864_2018-02-22T04:00:00Z
Buffered record: 17_399_2018-02-21T21:15:00Z
Buffered record: 17_35_2018-02-21T21:15:00Z
Buffered record: 17_394_2018-01-24T09:30:00Z
Buffered record: 17_36_2018-02-21T20:45:00Z
Buffered record: 17_392_2018-02-21T20:45:00Z
Buffered record: 17_393_2018-02-21T21:15:00Z
Buffered record: 17_12234783_2026-03-03T12:45:00Z
Buffered record: 17_12234785_2026-03-03T12:45:00Z
Buffered record: 17_12234787_2026-03-03T12:45:00Z
Buffered record: 17_12234784_2026-03-03T12:45:00Z
Buffered record: 17_12234786_2026-03-03T12:45:00Z
Buffered record: 17_12234782_2026-03-03T12:45:00Z
Buffered record: 17_12234790_2026-03-03T12:45:00Z
Buffered record: 17_12234788_2026-03-03T12:45:00Z
Buffered record: 17_12234789_2026-03-03T12:45:00Z
B

C:\Users\admin\AppData\Local\Temp\ipykernel_19004\2009816604.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Uploaded batch: raw/latest/openaq_20260309_164449.json
Buffered record: 178_296_2016-03-06T18:00:00Z
Buffered record: 178_295_2016-03-06T18:00:00Z
Buffered record: 179_297_2016-03-06T18:00:00Z
Buffered record: 179_298_2016-03-06T18:00:00Z
Buffered record: 180_300_2016-03-06T18:00:00Z
Buffered record: 180_299_2016-03-06T18:00:00Z
Buffered record: 181_301_2016-03-06T18:00:00Z
Buffered record: 183_333_2016-03-06T19:00:00Z
Buffered record: 183_305_2016-03-06T19:00:00Z
Buffered record: 184_306_2016-03-06T19:00:00Z
Buffered record: 186_309_2016-03-06T20:00:00Z
Buffered record: 186_310_2026-03-08T10:00:00Z
Buffered record: 186_2031_2026-03-07T11:00:00Z
Buffered record: 187_327_2016-03-06T20:00:00Z
Buffered record: 187_2071339_2026-03-07T11:00:00Z
Buffered record: 187_311_2026-03-08T10:00:00Z
Buffered record: 188_313_2016-03-06T20:00:00Z
Buffered record: 188_312_2026-03-08T10:00:00Z
Buffered record: 189_314_2016-03-06T19:00:00Z
Buffered record: 190_315_2016-03-06T19:00:00Z
Buffered record: 191


KeyboardInterrupt


KeyboardInterrupt

